# Price Fine-Tuning — Free Model (QLoRA)

Self-contained Colab notebook for the **"The Price is Right"** project.

**What this version does**
- Uses a **free public model** (`Qwen/Qwen2.5-1.5B-Instruct`) — no gated Llama, no HF approval needed
- Loads **your dataset** (`AbhishekRavindran/items_prompts_lite`) which already has `prompt` + `completion`
- Falls back to the public `ed-donner/items_prompts_lite` if needed
- Trains with 4-bit QLoRA on a free Colab T4
- Saves and optionally downloads the LoRA adapter

No ZIP uploads. No private `pricer` / `util` modules required.

## 1. Install dependencies

In [7]:
!pip -q uninstall -y transformers peft accelerate bitsandbytes
!pip -q install -U "transformers>=4.45,<5" "peft>=0.13" "accelerate>=0.34" "bitsandbytes>=0.46.1" datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 21.0 MB/s eta 0:00:00


In [1]:
import os
import re
import shutil
from pathlib import Path

import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. In Colab: Runtime → Change runtime type → T4 GPU, then restart from the top."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.6 GB


## 2. Configuration

- **Model**: free public Qwen 1.5B Instruct (no token required)
- **Dataset**: your lite set first, then public fallback
- Settings tuned for a free Colab T4

In [2]:
# Free public model (no gated access, no HF token needed for the model)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Your datasets (from day2 notebook). Lite is recommended for free Colab.
LITE_MODE = True
DATA_USER = "AbhishekRavindran"
PRIMARY_DATASET = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

# Public fallback if the primary is private / unavailable
FALLBACK_DATASET = "ed-donner/items_prompts_lite" if LITE_MODE else "ed-donner/pricer-data"

# Training settings for T4
MAX_LENGTH = 160
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
SEED = 42

# Optional caps (None = use full split). Lite already has 20k/1k/1k.
MAX_TRAIN_SAMPLES = 400
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = 200   # keep small for quick test at the end

OUTPUT_DIR = "/content/price_qlora"
ZIP_OUTPUT = "/content/price_qlora_adapter.zip"

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Model:", MODEL_NAME)
print("Primary dataset:", PRIMARY_DATASET)
print("Fallback dataset:", FALLBACK_DATASET)
print("Output:", OUTPUT_DIR)

Model: Qwen/Qwen2.5-1.5B-Instruct
Primary dataset: AbhishekRavindran/items_prompts_lite
Fallback dataset: ed-donner/items_prompts_lite
Output: /content/price_qlora


## 3. Load dataset

Tries your dataset first, then the public fallback. No file upload needed.

In [3]:
def try_load(name):
    print(f"Trying: {name}")
    return load_dataset(name)

ds = None
last_err = None
for name in (PRIMARY_DATASET, FALLBACK_DATASET):
    try:
        ds = try_load(name)
        print(f"Loaded: {name}")
        break
    except Exception as e:
        last_err = e
        print(f"  failed: {e}")

if ds is None:
    raise RuntimeError(f"Could not load any dataset. Last error: {last_err}")

print(ds)
print("\nColumns:", ds[list(ds.keys())[0]].column_names)
print("\nExample:")
print(ds[list(ds.keys())[0]][0])

Trying: AbhishekRavindran/items_prompts_lite
Loaded: AbhishekRavindran/items_prompts_lite
DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 20000
    })
    val: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1000
    })
})

Columns: ['prompt', 'completion']

Example:
{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.\n\nPrice is $', 'completion': '64.00'}


In [4]:
def to_rows(split_name, max_samples):
    if split_name not in ds:
        # some datasets use 'validation' instead of 'val'
        alt = {"val": "validation", "validation": "val"}.get(split_name)
        if alt and alt in ds:
            split_name = alt
        else:
            return None

    rows = []
    for ex in ds[split_name]:
        # Preferred: already has prompt + completion
        if "prompt" in ex and "completion" in ex:
            prompt = str(ex["prompt"]).strip()
            completion = str(ex["completion"]).strip()
        # Fallback for pricer-data style
        elif "text" in ex:
            text = ex["text"]
            if "Price is $" in text:
                prompt = text.split("Price is $")[0] + "Price is $"
                completion = text.split("Price is $")[1].strip()
            else:
                prompt = text
                completion = str(ex.get("price", ""))
        else:
            continue

        if prompt and completion:
            rows.append({"prompt": prompt, "completion": completion})

    df = pd.DataFrame(rows)
    if max_samples and len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=SEED).reset_index(drop=True)
    return df


# --- fixed loading logic (no ambiguous DataFrame truth value) ---
train_norm = to_rows("train", MAX_TRAIN_SAMPLES)

val_norm = to_rows("val", MAX_VAL_SAMPLES)
if val_norm is None:
    val_norm = to_rows("validation", MAX_VAL_SAMPLES)

test_norm = to_rows("test", MAX_TEST_SAMPLES)

if train_norm is None or len(train_norm) < 10:
    raise RuntimeError("Train split is empty or missing. Check the dataset columns above.")

if val_norm is None or len(val_norm) == 0:
    val_norm = train_norm.sample(n=min(1000, max(1, len(train_norm)//10)), random_state=SEED)
    train_norm = train_norm.drop(val_norm.index).reset_index(drop=True)
    val_norm = val_norm.reset_index(drop=True)

if test_norm is None or len(test_norm) == 0:
    test_norm = train_norm.sample(n=min(200, max(1, len(train_norm)//20)), random_state=SEED+1)
    train_norm = train_norm.drop(test_norm.index).reset_index(drop=True)
    test_norm = test_norm.reset_index(drop=True)

print(f"Train:      {len(train_norm):,}")
print(f"Validation: {len(val_norm):,}")
print(f"Test:       {len(test_norm):,}")
print("\nSample:")
display(train_norm.head(2))

Train:      400
Validation: 1,000
Test:       200

Sample:


,prompt,completion
0,What does this cost to the nearest dollar?\n\n...,79.00
1,What does this cost to the nearest dollar?\n\n...,20.00


In [5]:
# !pip -q install -U "bitsandbytes>=0.46.1" accelerate

## 4. Load tokenizer + QLoRA model

## 5. Tokenize

In [7]:
def build_text(prompt, completion):
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": completion},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


def tokenize_row(example):
    text = build_text(example["prompt"], example["completion"])
    encoded = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    labels = encoded["input_ids"].copy()
    labels = [t if m == 1 else -100 for t, m in zip(labels, encoded["attention_mask"])]
    encoded["labels"] = labels
    return encoded


train_ds = Dataset.from_pandas(train_norm[["prompt", "completion"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_norm[["prompt", "completion"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_norm[["prompt", "completion"]], preserve_index=False)

train_tok = train_ds.map(tokenize_row, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize_row, remove_columns=val_ds.column_names)
test_tok  = test_ds.map(tokenize_row, remove_columns=test_ds.column_names)

print(train_tok)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 400
})


## 6. Train

In [8]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
)

train_result = trainer.train()
print(train_result)

Step,Training Loss,Validation Loss


TrainOutput(global_step=50, training_loss=1.701704921722412, metrics={'train_runtime': 378.2575, 'train_samples_per_second': 1.057, 'train_steps_per_second': 0.132, 'total_flos': 510261264384000.0, 'train_loss': 1.701704921722412, 'epoch': 1.0})


## 7. Save adapter

In [9]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved files:")
for p in sorted(Path(OUTPUT_DIR).rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(OUTPUT_DIR))

Saved files:
 - README.md
 - adapter_config.json
 - adapter_model.safetensors
 - added_tokens.json
 - chat_template.jinja
 - checkpoint-50/README.md
 - checkpoint-50/adapter_config.json
 - checkpoint-50/adapter_model.safetensors
 - checkpoint-50/optimizer.pt
 - checkpoint-50/rng_state.pth
 - checkpoint-50/scheduler.pt
 - checkpoint-50/trainer_state.json
 - checkpoint-50/training_args.bin
 - merges.txt
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin
 - vocab.json


## 8. Quick test

In [10]:
model.config.use_cache = True
model.eval()

def predict(prompt, max_new_tokens=16):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


ex = test_norm.iloc[0]
print("PROMPT (truncated):")
print(ex["prompt"][:400] + ("..." if len(ex["prompt"]) > 400 else ""))
print("\nEXPECTED:", ex["completion"])
print("MODEL:   ", predict(ex["prompt"]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PROMPT (truncated):
What does this cost to the nearest dollar?

Title: RUNSY 5000 mAh Battery Case for Samsung Galaxy S20 5G  
Category: Electronics  
Brand: RUNSY  
Description: A slim 2‑in‑1 battery case that adds 105% extra battery life to the Samsung Galaxy S20 5G.  
Details: Features a 5000 mAh lithium‑polymer battery, built‑in short‑circuit and overcharge protection, a power switch with 4‑LED status indicator, ...

EXPECTED: 29.2
MODEL:    79.00


## 9. Zip + download adapter

In [12]:
if os.path.exists(ZIP_OUTPUT):
    os.remove(ZIP_OUTPUT)

shutil.make_archive(
    base_name=ZIP_OUTPUT.replace(".zip", ""),
    format="zip",
    root_dir=OUTPUT_DIR,
)
print("Created:", ZIP_OUTPUT)

Created: /content/price_qlora_adapter.zip


In [13]:
from google.colab import files
files.download(ZIP_OUTPUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
from google.colab import userdata
from huggingface_hub import login, HfApi

# Login
hf_token = userdata.get("HF_TOKEN")   # make sure you added this secret in Colab
login(hf_token)

# Change to your username + repo name
REPO_ID = "Charan2804/price-qlora-qwen-1.5b"

api = HfApi()
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True, private=False)

api.upload_folder(
    folder_path="/content/price_qlora",
    repo_id=REPO_ID,
    repo_type="model",
)

print(f"Uploaded to: https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...heckpoint-50/optimizer.pt:   0%|          |  130kB / 38.0MB            

  ...heckpoint-50/scheduler.pt: 100%|##########| 1.47kB / 1.47kB            

  ...eckpoint-50/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

...

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

  ...oint-50/training_args.bin:  24%|##4       | 1.41kB / 5.84kB            

  ...e_qlora/training_args.bin:  24%|##4       | 1.41kB / 5.84kB            

Uploaded to: https://huggingface.co/Charan2804/price-qlora-qwen-1.5b


## Notes

- **Model is free**: `Qwen/Qwen2.5-1.5B-Instruct` needs no Hugging Face approval.
- **Dataset**: uses your `AbhishekRavindran/items_prompts_lite` (or the public fallback).
- For a longer run set `NUM_EPOCHS = 2` and/or switch `LITE_MODE = False` (needs more time/GPU).
- OOM? Lower `MAX_LENGTH` to 160 and `TRAIN_BATCH_SIZE` to 1.
- To load the adapter later:

```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quant_config, device_map="auto")
model = PeftModel.from_pretrained(base, "/content/price_qlora")
```